# King County: from preprocessing to a saved model

First compare preprocessing with the original baseline, then train a separate complete pipeline on a train/test split. Training, evaluation, and export stay in this notebook—not in a training service.

Run from the repository root with the project's `.venv` kernel. Export creates a new local bundle; it never overwrites `model/model.bin`. See [the baseline reference](docs/notebook-baseline.md).


In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline

from king_county.cleaning import HouseCleaner
from king_county.features import HouseFeatureEngineer

## Load the original dataset

In [ ]:
# Load the dataset with pandas
kc_data = pd.read_csv("data/King_County_House_prices_dataset.csv")

## Training-only row exclusion

In [ ]:
# Known exclusion in this dataset only; not part of prediction preprocessing.
kc_data = kc_data.drop(index=15856)

## Reusable cleaning and feature pipeline

`HouseCleaner` applies fixed rules without learning values. `HouseFeatureEngineer.fit()` stores a copy of waterfront coordinates; `transform()` creates both distances and the 19 ordered model inputs using those stored locations. Price, ID, date, and any supplied distances are not passed through.

**Baseline comparison only:** we fit on all retained rows here to match the original notebook. For model evaluation, split first and fit this pipeline on training rows only. The modelling section below builds a separate pipeline, evaluates it, and exports it. Do not use these full-data fitted references for a held-out test.

In [ ]:
preprocessor = Pipeline(
    [
        ("cleaning", HouseCleaner()),
        ("features", HouseFeatureEngineer()),
    ]
)
X_prepared = preprocessor.fit_transform(kc_data)
X_prepared.head()

## Prepared features

The output keeps row labels and returns the same 19 input names in a fixed order. The feature object retains only waterfront coordinates, not complete training records or sale prices. No model is trained or overwritten.

In [ ]:
engineer = preprocessor.named_steps["features"]
print(f"Prepared rows: {len(X_prepared)}; features: {X_prepared.shape[1]}")
print(f"Stored waterfront references: {len(engineer.waterfront_locations_)}")
print(f"Missing prepared values: {X_prepared.isna().sum().sum()}")

## Train a separate complete pipeline

The full-data `preprocessor` above is for regression comparison only. Do not reuse it for evaluation. From here onward, split the **raw inputs** first and put fresh cleaning and feature objects inside the model pipeline.

We keep the original degree-2 polynomial features, input scaling, target scaling, ElasticNet settings, and 9-candidate / 5-fold grid search. The intentional difference is that each training fold learns its own waterfront coordinates. Price is the target: **total sale price in US dollars**, not price per square foot.

The shuffled split is an educational baseline. Repeated house IDs can cross splits; this does not establish performance on future sales or entirely unseen properties. Do not tune against the held-out test score.

In [ ]:
import hashlib
import json
import logging
import platform
import subprocess
import sys
from datetime import UTC, datetime
from importlib.metadata import version
from pathlib import Path

import numpy as np
import skops.io as sio
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from king_county.logging_config import configure_logging
from king_county.settings import AppSettings

configure_logging(AppSettings.from_environment())
logger = logging.getLogger("king_county.notebook")

In [ ]:
RAW_INPUT_COLUMNS = [
    "bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors",
    "waterfront", "view", "condition", "grade", "sqft_above",
    "yr_built", "yr_renovated", "zipcode", "lat", "long",
    "sqft_living15", "sqft_lot15",
]
dataset_path = Path("data/King_County_House_prices_dataset.csv").resolve()
source_sha256 = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
X_raw = kc_data.loc[:, RAW_INPUT_COLUMNS].copy()
y = kc_data["price"].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.3, random_state=42,
)
logger.info("Split raw houses: %d train, %d test", len(X_train), len(X_test))
print(f"Raw input fields: {X_raw.shape[1]}; target: total sale price (USD)")

### Build and tune

`cleaning → features → polynomial → scaler → model`

`GridSearchCV` clones this entire pipeline for every fold. No test rows or validation-fold rows enter the corresponding training waterfront reference list. The winning pipeline is refitted on all training rows, never on the held-out test set.

Expect the next fit cell to take longer than the ordinary test suite: it runs 45 fits plus the final refit. It uses one worker to keep resource use bounded. CI only uses a small controlled example.

In [ ]:
model_pipeline = Pipeline([
    ("cleaning", HouseCleaner()),
    ("features", HouseFeatureEngineer()),
    ("polynomial", PolynomialFeatures(2, include_bias=False)),
    ("scaler", StandardScaler()),
    ("model", TransformedTargetRegressor(
        regressor=ElasticNet(max_iter=5000, tol=1e-4, precompute=True),
        transformer=StandardScaler(),
    )),
])
param_grid = {
    "model__regressor__alpha": [0.01, 0.1, 1],
    "model__regressor__l1_ratio": [0.2, 0.5, 0.8],
}
search = GridSearchCV(
    model_pipeline, param_grid, cv=5, scoring="r2",
    verbose=2, n_jobs=1, error_score="raise",
)

In [ ]:
logger.info("Starting training-only cross-validation")
search.fit(X_train, y_train)
fitted_pipeline = search.best_estimator_
print("Best parameters:", search.best_params_)
print(f"Mean cross-validation R²: {search.best_score_:.4f}")

In [ ]:
predictions = fitted_pipeline.predict(X_test)
if not np.isfinite(predictions).all():
    raise RuntimeError("Evaluation produced non-finite predictions.")
metrics = {
    "r2": float(r2_score(y_test, predictions)),
    "mae_usd": float(mean_absolute_error(y_test, predictions)),
    "rmse_usd": float(root_mean_squared_error(y_test, predictions)),
}
dummy = DummyRegressor(strategy="mean").fit(X_train, y_train)
baseline_metrics = {
    "r2": float(dummy.score(X_test, y_test)),
    "mae_usd": float(mean_absolute_error(y_test, dummy.predict(X_test))),
}
if not all(np.isfinite(value) for value in metrics.values()):
    raise RuntimeError("Evaluation produced non-finite metrics.")
print(json.dumps({"test": metrics, "mean_baseline": baseline_metrics}, indent=2))
print("Training waterfront references:",
      len(fitted_pipeline.named_steps["features"].waterfront_locations_))

### Read the evaluation carefully

- **R²:** performance relative to always predicting the test-set mean; it is not percentage accuracy.
- **MAE:** average absolute price error, in dollars.
- **RMSE:** a dollar error measure that gives larger misses more weight.
- **Mean baseline:** a simple predictor fitted only on training prices.

These scores describe this held-out split, not a guarantee for individual houses. Inspect the results before approving a model; export is not deployment or a quality approval. A different training boundary means we must not reuse the original notebook's historical R² claim.

## Export a new local bundle

Run the following cells only after reviewing the evaluation. Each run creates a new timestamped directory under ignored `model/refactored/`:

- `pipeline.skops`: cleaning, fitted waterfront references, polynomial expansion, both scalers, and the trained regressor.
- `metadata.json`: checksum, input/output names, data fingerprint, split/tuning settings, measured metrics, and dependency versions.
- `reload-example.json`: three held-out feature-only records and their expected predictions, for a fresh-process check.

The example includes locations from this public teaching dataset; exported bundles remain local and ignored. The existing `model/model.bin` is not touched.

Only our two named custom transformer types are allowed when reloading. Do not automatically trust every type reported by skops. The checksum recorded here identifies the new artifact for review; the future API must pin an approved checksum in code, not trust arbitrary metadata.

In [ ]:
model_bytes = sio.dumps(fitted_pipeline)
model_sha256 = hashlib.sha256(model_bytes).hexdigest()
created_at = datetime.now(UTC)
model_version = created_at.strftime("king-county-%Y%m%dT%H%M%S%fZ")
metadata = {
    "model_version": model_version,
    "created_at_utc": created_at.isoformat(),
    "model_sha256": model_sha256,
    "target": "price", "currency": "USD",
    "raw_input_columns": RAW_INPUT_COLUMNS,
    "prepared_feature_columns": list(HouseFeatureEngineer.FEATURE_NAMES),
    "dataset_sha256": source_sha256,
    "training_row_exclusion": {"index": 15856, "reason": "Original notebook rule"},
    "split": {"test_size": 0.3, "random_state": 42,
              "train_rows": len(X_train), "test_rows": len(X_test)},
    "cross_validation": {
        "folds": search.cv, "scoring": search.scoring,
        "best_mean_r2": float(search.best_score_),
        "best_params": search.best_params_,
        "param_grid": search.param_grid,
    },
    "test_metrics": metrics,
    "mean_baseline_metrics": baseline_metrics,
    "waterfront_reference_count":
        len(fitted_pipeline.named_steps["features"].waterfront_locations_),
    "versions": {"python": platform.python_version(), **{
        name: version(name) for name in
        ["mle-refactoring-project", "numpy", "pandas", "scikit-learn", "skops"]
    }},
    "limitations": [
        "2014–2015 sales; not a current valuation service",
        "Shuffled row split; repeated properties may cross splits",
        "Not evaluated on future sales or entirely unseen properties",
    ],
}
reload_example = {
    "inputs": json.loads(X_test.head(3).to_json(orient="records")),
    "predictions": predictions[:3].tolist(),
}

In [ ]:
export_directory = Path("model/refactored") / model_version
export_directory.mkdir(parents=True, exist_ok=False)
artifact_path = export_directory / "pipeline.skops"
with artifact_path.open("xb") as artifact:
    artifact.write(model_bytes)
for filename, content in (
    ("metadata.json", metadata), ("reload-example.json", reload_example),
):
    with (export_directory / filename).open("x") as output:
        json.dump(content, output, indent=2, ensure_ascii=False, allow_nan=False)
        output.write("\n")
logger.info("Exported new model bundle: %s", export_directory)
print("New artifact:", artifact_path)
print("SHA-256 for review:", model_sha256)

### Verify without notebook state or the CSV

A new Python process receives only the three exported files. It runs from a temporary directory, verifies the artifact bytes, checks the explicit skops type allowlist, loads those same bytes, and compares predictions.

This checks serialization and reproducibility, not accuracy. The check fails on a checksum mismatch, unexpected type, changed predictions, missing dependency, or incompatible package. It does not automatically approve the model for the API.

In [ ]:
reload_script = """
import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import skops.io as sio

artifact = Path(sys.argv[1])
metadata = json.loads(artifact.with_name("metadata.json").read_text())
example = json.loads(artifact.with_name("reload-example.json").read_text())
data = artifact.read_bytes()
if hashlib.sha256(data).hexdigest() != sys.argv[2]:
    raise RuntimeError("Model checksum mismatch.")
allowed = [
    "king_county.cleaning.HouseCleaner",
    "king_county.features.HouseFeatureEngineer",
]
unexpected = set(sio.get_untrusted_types(data=data)) - set(allowed)
if unexpected:
    raise RuntimeError(f"Unexpected model types: {sorted(unexpected)}")
pipeline = sio.loads(data, trusted=allowed)
houses = pd.DataFrame(example["inputs"], columns=metadata["raw_input_columns"])
actual = pipeline.predict(houses)
np.testing.assert_allclose(actual, example["predictions"], rtol=1e-10, atol=1e-6)
print(f"Fresh-process reload passed for {len(actual)} houses.")
"""

In [ ]:
from tempfile import TemporaryDirectory

with TemporaryDirectory(prefix="king-county-reload-") as isolated_directory:
    reloaded = subprocess.run(
        [sys.executable, "-c", reload_script,
         str(artifact_path.resolve()), model_sha256],
        cwd=isolated_directory, capture_output=True, text=True,
        timeout=60, check=True,
    )
print(reloaded.stdout.strip())
logger.info("Fresh-process artifact verification succeeded")